In [ ]:
import pandas as pd

match = pd.read_csv(
    r"Projects\ADHD2\Data Availiabity\valid\Answer_file.csv"
)

A = pd.read_csv(
    r"Projects\ADHD2\Data\Aset.csv"
)

B = pd.read_csv(
    r"Projects\ADHD2\Data\Bset.csv"
)

matched = (
    match
    .sort_values("best_similarity", ascending=False)
    .drop_duplicates("best_a_id")
    .copy()
)

matched["a_idx"] = (
    matched["best_a_id"]
    .astype(str)
    .str.replace("A_", "", regex=False)
    .astype(int)
)

matched["b_idx"] = (
    matched["b_id"]
    .astype(str)
    .str.replace("B_", "", regex=False)
    .astype(int)
)

A_selected = A.loc[
    matched["a_idx"],
    ["QuestionA", "AnswerA"]
].reset_index()

A_selected = A_selected.rename(
    columns={
        "index": "a_idx",
        "QuestionA": "a_question",
        "AnswerA": "a_answer",
    }
)

B_selected = B.loc[
    matched["b_idx"],
    ["question_full", "answer_full"]
].reset_index()

B_selected = B_selected.rename(
    columns={
        "index": "b_idx",
        "question_full": "b_question",
        "answer_full": "b_answer",
    }
)

result = (
    matched[["best_a_id", "b_id", "a_idx", "b_idx", "best_similarity"]]
    .merge(A_selected, on="a_idx", how="left")
    .merge(B_selected, on="b_idx", how="left")
)

result.to_csv(
    r"Projects\ADHD2\Data\Matched_62_QA.csv",
    index=False,
    encoding="utf-8-sig"
)

print(result.head())
print(f"saved rows: {len(result)}")

In [ ]:
import os
import json
import re
import time
import pandas as pd
import requests
from pathlib import Path
from dotenv import load_dotenv

try:
    import dirtyjson
except ImportError:
    dirtyjson = None

load_dotenv()

INPUT_CSV = r"Projects\ADHD2\Data\Matched_QA.csv"
OUTPUT_CSV = r"Projects\ADHD2\Data\Matched_QA_coded_results.csv"
OUTPUT_JSON_DIR = r"Projects\ADHD2\Data\Matched_QA_json_outputs"
OUTPUT_JSONL = r"Projects\ADHD2\Data\Matched_QA_coded_results_raw.jsonl"

API_KEY = os.getenv("OPENAI_API_KEY")
BASE_URL = "https://api.openai.com/v1"
MODEL = "gpt-4o-mini"

if not API_KEY:
    raise ValueError("OPENAI_API_KEY가 없습니다. 환경변수 또는 .env 파일에 설정하세요.")

Path(OUTPUT_JSON_DIR).mkdir(parents=True, exist_ok=True)


SYSTEM_PROMPT = """
당신은 ADHD 건강정보 응답을 대상으로 하는 보조적(framework-guided) 연역적 내용분석을 수행하는 연구 보조자입니다.
반드시 주어진 코드북에 따라 분류하고, JSON 이외의 텍스트는 출력하지 마십시오.
"""

def build_user_prompt(pair_id, answer_a, answer_b):
    return f"""
당신은 ADHD 건강정보 응답을 대상으로 하는 보조적(framework-guided) 연역적 내용분석을 수행하는 연구 보조자입니다.

이 분석의 목적은 소비자 관점에서 관찰 가능한 커뮤니케이션 특성을 분류하는 것입니다.

다음 사항을 반드시 준수하십시오.

1. 의학적 정확성이나 임상적 적절성을 평가하지 마십시오.
2. 답변의 옳고 그름을 판단하지 마십시오.
3. 명시적으로 표현되지 않은 의도, 감정, 근거, 결과 또는 행동권고를 추론하지 마십시오.
4. 분석 단위는 답변 전체입니다.
5. 각 영역은 서로 독립적으로 코딩하십시오.
6. 각 영역마다 반드시 하나의 범주만 선택하십시오.
7. 모든 판단은 답변에 명시적으로 표현된 내용만 근거로 하십시오.
8. 분류 근거가 되는 문장을 그대로 supporting_excerpt에 제시하십시오.
9. 두 범주 사이에서 판단이 어려우면 confidence를 low로 표시하십시오.

──────────────────────────────
영역 1. 의사소통 기능
──────────────────────────────

① 정보제공
답변의 주된 목적이 사실적, 개념적, 인과적 설명이나 선택지, 절차, 행동 가능한 정보를 제공하는 것이다.

② 정서적 지지
답변의 주된 목적이 감정의 인정, 위로, 안심, 격려 등 정서적 지지를 제공하는 것이다.

③ 경험 중심 설명
답변이 실제 환자, 보호자, 의료인 또는 개인의 경험이나 사례를 중심으로 설명된다.

※ 가상의 사례는 경험 중심 설명으로 분류하지 않는다.

④ 혼합 또는 기타
하나의 범주가 명확하게 우세하지 않다.

──────────────────────────────
영역 2. 근거 및 권위 제시
──────────────────────────────

① 연구근거형
연구논문, 과학문헌, 임상진료지침, 공공보건기관 또는 연구결과를 근거로 명시적으로 제시한다.

② 전문가권위형
연구근거 없이 전문가 의견, 임상경험, 전문가 권위 또는 전문가 합의를 명시적으로 제시한다.

③ 둘 다
연구근거와 전문가 권위가 모두 명시적으로 제시된다.

④ 없음
연구근거와 전문가 권위가 모두 명시적으로 제시되지 않는다.

※ "의사와 상담하세요." 또는 "약사와 상담하세요."와 같은 단순 권고는 전문가권위형으로 분류하지 않는다.

──────────────────────────────
영역 3. 결과 프레이밍
──────────────────────────────

① 이득강조형
권장 행동을 했을 때 얻는 이득, 개선, 긍정적 결과 또는 위험 감소를 주로 강조한다.

② 위험강조형
권장 행동을 하지 않았을 때의 손실, 위해, 악화 또는 합병증을 주로 강조한다.

③ 균형형
이득과 위험을 비교적 균형 있게 함께 제시한다.

④ 결과 제시 없음
행동이나 선택의 결과를 명시적으로 설명하지 않는다.

──────────────────────────────
영역 4. 행동권고 및 실행가능성
──────────────────────────────

① 구체적 행동권고
소비자가 수행할 수 있는 구체적인 행동을 제시하며, 언제, 어떻게, 어떤 조건에서 수행해야 하는지 명확히 설명한다.

② 일반 행동권고
건강 관련 행동을 권고하지만 수행 방법이나 시기, 조건에 대한 설명은 부족하다.

③ 의료기관 의뢰만 제시
의료기관, 의사, 약사, 응급실 등을 방문하거나 상담하도록만 권고하며, 소비자가 직접 수행할 수 있는 다른 행동은 제시하지 않는다.

④ 행동권고 없음
행동을 권고하지 않고 설명, 평가, 경험 공유 또는 정서적 지지만 제공한다.

※ 단, "가슴통증이 생기면 즉시 응급실로 가십시오."처럼 시기와 조건이 명확한 경우에는 '구체적 행동권고'로 분류한다.

──────────────────────────────

Provider-generated response

{answer_a}

Community response

{answer_b}

다음 JSON 형식만 출력하십시오.

{{
  "pair_id": "{pair_id}",
  "provider": {{
    "communicative_function": {{
      "category": "",
      "supporting_excerpt": [""],
      "rationale": "",
      "confidence": "high | medium | low"
    }},
    "evidence_authority_signaling": {{
      "category": "",
      "research_evidence_excerpt": [],
      "professional_authority_excerpt": [],
      "rationale": "",
      "confidence": "high | medium | low"
    }},
    "outcome_framing": {{
      "category": "",
      "supporting_excerpt": [""],
      "rationale": "",
      "confidence": "high | medium | low"
    }},
    "behavioral_recommendations_actionability": {{
      "category": "",
      "supporting_excerpt": [""],
      "rationale": "",
      "confidence": "high | medium | low"
    }}
  }},
  "community": {{
    "communicative_function": {{
      "category": "",
      "supporting_excerpt": [""],
      "rationale": "",
      "confidence": "high | medium | low"
    }},
    "evidence_authority_signaling": {{
      "category": "",
      "research_evidence_excerpt": [],
      "professional_authority_excerpt": [],
      "rationale": "",
      "confidence": "high | medium | low"
    }},
    "outcome_framing": {{
      "category": "",
      "supporting_excerpt": [""],
      "rationale": "",
      "confidence": "high | medium | low"
    }},
    "behavioral_recommendations_actionability": {{
      "category": "",
      "supporting_excerpt": [""],
      "rationale": "",
      "confidence": "high | medium | low"
    }}
  }}
}}
"""

def call_llm(system_prompt, user_prompt):
    url = f"{BASE_URL}/chat/completions"

    headers = {
        "Authorization": f"Bearer {API_KEY}",
        "Content-Type": "application/json",
    }

    payload = {
        "model": MODEL,
        "temperature": 0,
        "messages": [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt},
        ],
    }

    response = requests.post(url, headers=headers, json=payload, timeout=120)
    response.raise_for_status()

    content = response.json()["choices"][0]["message"].get("content")
    if not content:
        raise ValueError("LLM 응답 content가 비어 있습니다.")

    return content

def extract_json(text):
    text = text.strip()

    if text.startswith("```"):
        text = text.replace("```json", "").replace("```", "").strip()

    match = re.search(r"\{.*\}", text, re.DOTALL)
    if not match:
        raise ValueError("JSON을 찾을 수 없습니다.")

    json_text = match.group(0)

    try:
        return json.loads(json_text)
    except json.JSONDecodeError:
        if dirtyjson is not None:
            return dirtyjson.loads(json_text)
        raise

def safe_filename(name):
    return re.sub(r'[\\/*?:"<>|]+', "_", str(name)).strip()

def as_json_text(value):
    return json.dumps(value, ensure_ascii=False)

def flatten_domain(parsed, side, domain):
    data = parsed[side][domain]

    row = {
        f"{side}_{domain}_category": data.get("category", ""),
        f"{side}_{domain}_rationale": data.get("rationale", ""),
        f"{side}_{domain}_confidence": data.get("confidence", ""),
    }

    if domain == "evidence_authority_signaling":
        row[f"{side}_{domain}_research_evidence_excerpt"] = as_json_text(
            data.get("research_evidence_excerpt", [])
        )
        row[f"{side}_{domain}_professional_authority_excerpt"] = as_json_text(
            data.get("professional_authority_excerpt", [])
        )
    else:
        row[f"{side}_{domain}_supporting_excerpt"] = as_json_text(
            data.get("supporting_excerpt", [])
        )

    return row

def flatten_result(parsed):
    row = {"pair_id": parsed.get("pair_id", "")}

    domains = [
        "communicative_function",
        "evidence_authority_signaling",
        "outcome_framing",
        "behavioral_recommendations_actionability",
    ]

    for side in ["provider", "community"]:
        for domain in domains:
            row.update(flatten_domain(parsed, side, domain))

    return row

def main():
    df = pd.read_csv(INPUT_CSV, encoding="utf-8-sig")
    df.columns = df.columns.str.strip()

    required = ["best_a_id", "b_id", "a_answer", "b_answer"]
    missing = [c for c in required if c not in df.columns]
    if missing:
        raise KeyError(f"필수 컬럼이 없습니다: {missing}\n현재 컬럼: {df.columns.tolist()}")

    results = []

    with open(OUTPUT_JSONL, "w", encoding="utf-8") as jsonl_f:
        for idx, row in df.iterrows():
            pair_id = f"{row['best_a_id']}-{row['b_id']}"
            print(f"Processing {pair_id}")

            user_prompt = build_user_prompt(
                pair_id=pair_id,
                answer_a=str(row["a_answer"]),
                answer_b=str(row["b_answer"]),
            )

            try:
                raw_output = call_llm(SYSTEM_PROMPT, user_prompt)
                parsed = extract_json(raw_output)

                json_path = os.path.join(
                    OUTPUT_JSON_DIR,
                    f"{safe_filename(pair_id)}.json"
                )

                with open(json_path, "w", encoding="utf-8") as f:
                    json.dump(parsed, f, ensure_ascii=False, indent=2)

                jsonl_f.write(json.dumps(parsed, ensure_ascii=False) + "\n")

                flat = flatten_result(parsed)
                flat["best_a_id"] = row["best_a_id"]
                flat["b_id"] = row["b_id"]
                flat["best_similarity"] = row.get("best_similarity", "")
                results.append(flat)

            except Exception as e:
                print(f"Error in {pair_id}: {e}")
                continue

            time.sleep(0.5)

    result_df = pd.DataFrame(results)
    result_df.to_csv(OUTPUT_CSV, index=False, encoding="utf-8-sig")

    print("완료:", OUTPUT_CSV)
    print("JSON 폴더:", OUTPUT_JSON_DIR)
    print("JSONL:", OUTPUT_JSONL)

if __name__ == "__main__":
    main()

In [ ]:
import pandas as pd
from pathlib import Path

input_path = Path(r"Projects\ADHD2\Data\Matched_QA_coded_results.csv")

df = pd.read_csv(input_path, encoding="utf-8-sig")
df.columns = df.columns.str.strip()

print("Loaded file:", input_path)
print("Shape:", df.shape)
print("Columns:")
for c in df.columns:
    print(repr(c))

print("\nCategory columns:")
for c in df.columns:
    if "category" in c:
        print(repr(c))
        
import numpy as np
import matplotlib.pyplot as plt

from pathlib import Path
from scipy.stats import chi2_contingency

output_dir = Path(r"Projects\ADHD2\Data\stats_outputs")
output_dir.mkdir(parents=True, exist_ok=True)

provider_category_cols = [
    c for c in df.columns
    if c.startswith("provider_") and c.endswith("_category")
]

domains = []

for provider_col in provider_category_cols:
    domain = provider_col.replace("provider_", "").replace("_category", "")
    community_col = f"community_{domain}_category"

    if community_col in df.columns:
        domains.append({
            "domain": domain,
            "provider_col": provider_col,
            "community_col": community_col,
        })

print("Detected domains:")
for d in domains:
    print(d)

if not domains:
    raise ValueError("provider/community category 컬럼 쌍을 찾지 못했습니다.")

In [ ]:
import textwrap
from pathlib import Path

import matplotlib.patches as mpatches
import matplotlib.pyplot as plt
import pandas as pd


def find_project_root():
    candidates = []

    if "__file__" in globals():
        candidates.append(Path(__file__).resolve().parent)

    candidates.append(Path.cwd())

    for start_path in candidates:
        for path in [start_path, *start_path.parents]:
            stats_file = path / "Data" / "stats_outputs" / "descriptive_statistics_by_domain.csv"
            if stats_file.exists():
                return path

    raise FileNotFoundError(
        "Could not find Data/stats_outputs/descriptive_statistics_by_domain.csv. "
        "Run this script from inside the ADHD2 project folder."
    )


base_dir = find_project_root()
output_dir = base_dir / "Data" / "stats_outputs"
desc_df = pd.read_csv(output_dir / "descriptive_statistics_by_domain.csv")

desc_df["domain"] = desc_df["domain"].replace({
    "behavioral_recommendations_actionability": "actionability",
})

plt.rcParams.update({
    "font.family": "Times New Roman",
    "axes.unicode_minus": False,
    "figure.dpi": 300,
    "axes.linewidth": 0.8,
    "axes.edgecolor": "#333333",
    "xtick.color": "#333333",
    "ytick.color": "#333333",
})

domain_order = [
    "communicative_function",
    "evidence_authority_signaling",
    "outcome_framing",
    "actionability",
]

domain_labels = {
    "communicative_function": "Communicative Function",
    "evidence_authority_signaling": "Evidence and Authority Signaling",
    "outcome_framing": "Outcome Framing",
    "actionability": "Actionability",
}

category_labels = {
    "\uc815\ubcf4\uc81c\uacf5": "Information Provision",
    "\uc815\uc11c\uc801 \uc9c0\uc9c0": "Emotional Support",
    "\uacbd\ud5d8 \uc911\uc2ec \uc124\uba85": "Experience-Based Explanation",
    "\uc5f0\uad6c\uadfc\uac70\ud615": "Research Evidence",
    "\uc804\ubb38\uac00\uad8c\uc704\ud615": "Professional Authority",
    "\uc5c6\uc74c": "None",
    "\uc774\ub4dd\uac15\uc870\ud615": "Gain-Framed",
    "\uc704\ud5d8\uac15\uc870\ud615": "Risk-Framed",
    "\uacb0\uacfc \uc81c\uc2dc \uc5c6\uc74c": "No Outcome Framing",
    "\uade0\ud615\ud615": "Balanced",
    "\uad6c\uccb4\uc801 \ud589\ub3d9\uad8c\uace0": "Specific Action Recommendation",
    "\uc77c\ubc18 \ud589\ub3d9\uad8c\uace0": "General Action Recommendation",
    "\ud589\ub3d9\uad8c\uace0 \uc5c6\uc74c": "No Action Recommendation",
    "\uc758\ub8cc\uae30\uad00 \uc758\ub8b0\ub9cc \uc81c\uc2dc": "Medical Referral Only",
}

source_labels = {
    "Provider-generated response": "Expert",
    "Community response": "Online Community",
}

domain_colors = {
    "communicative_function": ("#B22222", "#F4A6A6"),
    "evidence_authority_signaling": ("#1F4E79", "#9DC3E6"),
    "outcome_framing": ("#3A7D44", "#A9D18E"),
    "actionability": ("#C49A00", "#FFE699"),
}


def wrap_label(label, width=16):
    return "\n".join(textwrap.wrap(str(label), width=width))


fig, axes = plt.subplots(2, 2, figsize=(18, 11))
axes = axes.flatten()
panel_letters = ["A", "B", "C", "D"]

for ax, domain, panel_letter in zip(axes, domain_order, panel_letters):
    plot_df = desc_df[desc_df["domain"] == domain].copy()
    plot_df["category_en"] = plot_df["category"].map(category_labels).fillna(plot_df["category"])
    plot_df["source_en"] = plot_df["source"].map(source_labels).fillna(plot_df["source"])

    pivot = plot_df.pivot_table(
        index="category_en",
        columns="source_en",
        values="percent",
        fill_value=0,
        sort=False,
    )

    if pivot.empty:
        raise ValueError(f"No data found for domain: {domain}")

    pivot = pivot.reindex(columns=["Expert", "Online Community"], fill_value=0)

    x_positions = range(len(pivot.index))
    bar_width = 0.36
    expert_color, community_color = domain_colors[domain]

    expert_bars = ax.bar(
        [x - bar_width / 2 for x in x_positions],
        pivot["Expert"],
        width=bar_width,
        color=expert_color,
        edgecolor="white",
        linewidth=0.8,
        label="Expert",
    )
    community_bars = ax.bar(
        [x + bar_width / 2 for x in x_positions],
        pivot["Online Community"],
        width=bar_width,
        color=community_color,
        edgecolor="white",
        linewidth=0.8,
        label="Online Community",
    )

    ax.bar_label(
        expert_bars,
        labels=[f"{value:.1f}%" if value > 0 else "" for value in pivot["Expert"]],
        padding=3,
        fontsize=8,
    )
    ax.bar_label(
        community_bars,
        labels=[f"{value:.1f}%" if value > 0 else "" for value in pivot["Online Community"]],
        padding=3,
        fontsize=8,
    )

    ax.set_title(
        f"({panel_letter}) {domain_labels[domain]}",
        fontsize=13.5,
        fontweight="bold",
        pad=14,
    )
    ax.set_xlabel("")
    ax.set_ylabel("Percentage (%)", fontsize=11)
    ax.set_ylim(0, 100)
    ax.set_xticks(list(x_positions))
    ax.set_xticklabels([wrap_label(label) for label in pivot.index], rotation=0, fontsize=9)
    ax.set_yticks([0, 25, 50, 75, 100])
    ax.grid(axis="y", linestyle="--", linewidth=0.6, alpha=0.28)
    ax.set_axisbelow(True)

    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

legend_handles = [
    mpatches.Patch(color="#444444", label="Expert (darker tone)"),
    mpatches.Patch(color="#BBBBBB", label="Online Community (lighter tone)"),
]

fig.legend(
    handles=legend_handles,
    loc="upper center",
    ncol=2,
    frameon=False,
    fontsize=11,
    bbox_to_anchor=(0.5, 0.985),
)

fig.suptitle(
    "LLM Assistant Content Analysis: A Comparison Between Information Provider and Online Community Answer Styles",
    fontsize=18,
    fontweight="bold",
    y=1.035,
)

plt.tight_layout(rect=[0.02, 0.02, 0.98, 0.94], h_pad=5.0)

output_fig = output_dir / "category_distribution_all_domains_grouped_bar_english.png"
plt.savefig(output_fig, dpi=300, bbox_inches="tight")
plt.close(fig)

print(f"saved: {output_fig}")
